In [ ]:
from notebook.services.config import ConfigManager
cm = ConfigManager()
cm.update('livereveal', {
    'width': 1920,
    'height': 1080,
    'scroll': True,
})

# Week 11: Wednesday, AST 5011: Astrophysical Systems

## Galaxy-Halo Coevolution & Feedback

### Michael Coughlin

Reference: Cimatti, Fraternali & Nipoti, Ch. 8

With material from Benedikt Diemer (UMD) and Frank van den Bosch (Yale).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator

from colossus.cosmology import cosmology
from colossus.lss import mass_function

import routines as rt

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

cosmo = cosmology.setCosmology('planck18')

## A Semi-Analytic Model of Galaxy Formation

We now bring together the physics from the previous lectures into a semi-analytic model (SAM) that tracks the co-evolution of four components:

1. Halo mass $M_h(t)$ — grows via accretion (Neistein+2008)
2. Gas mass $M_g(t)$ — accretes with baryons, depleted by star formation and winds
3. Stellar mass $M_*(t)$ — grows via star formation
4. Metal mass $M_Z(t)$ — enriched by stellar yields, diluted by inflow

The evolution is governed by coupled ODEs:

$$
\frac{dM_g}{dt} = \dot{M}_h \cdot f_b \cdot \epsilon_1 \cdot \epsilon_2 - \dot{M}_* \cdot (1 - R + \eta)
$$

$$
\frac{dM_*}{dt} = \text{SFR} \cdot (1 - R)
$$

where:

- $f_b = \Omega_b/\Omega_m$ is the cosmic baryon fraction
- $\epsilon_1$ is the reionization suppression factor (low-mass halos)
- $\epsilon_2$ is the cooling efficiency factor (high-mass halos)
- $R = 0.46$ is the mass return fraction from stellar evolution
- $\eta$ is the mass loading factor from stellar feedback / winds
- SFR $= M_g / t_{\text{depl}}$ (gas depletion time model)

## Key Physics Components

| Component | Effect | Typical Scale |
|---|---|---|
| Reionization ($\epsilon_1$) | Suppresses gas accretion in small halos after UV background heats IGM | $M_h < 10^{10}\,M_\odot$ |
| Cooling cutoff ($\epsilon_2$) | Prevents gas cooling in massive halos (long $t_{\text{cool}}$) | $M_h > 10^{12}\,M_\odot$ |
| Star formation | Converts gas to stars on depletion timescale $t_{\text{depl}} \sim 2\,$Gyr | All masses |
| Feedback/winds ($\eta$) | Ejects gas from galaxy; stronger in low-mass halos | $\eta \sim 2$–$300$ |

Together, these create the characteristic shape of the stellar-to-halo mass relation (SHMR): peak efficiency at $M_h \sim 10^{12}\,M_\odot$, suppressed at low masses (reionization + feedback) and high masses (cooling cutoff).

## Demonstration: Gas Accretion Efficiency

The combined gas accretion efficiency $\epsilon_1 \times \epsilon_2$ defines a "Goldilocks zone" of halo masses where gas can efficiently accrete and cool:

- $\epsilon_1$ (reionization): suppresses accretion for $M_h \lesssim 10^{10}\,M_\odot$ after reionization
- $\epsilon_2$ (cooling cutoff): suppresses cooling for $M_h \gtrsim 10^{12}\,M_\odot$

The product $\epsilon_1 \times \epsilon_2$ peaks near $M_h \sim 10^{11}\,M_\odot$ at $z = 0$, and this window narrows at lower redshifts as reionization bites harder and the cooling mass evolves. At high $z$ before reionization, only the cooling cutoff matters (adapted from CFN Chapter 8).

In [ ]:
# Gas accretion efficiency: eps_g1 * eps_g2 vs halo mass at multiple redshifts
from colossus.halo import mass_so

Mh_eps = 10**np.linspace(6, 15, 300)

# Model parameters (matching GalaxyModel defaults)
M_c1_0 = 2e10
M_c1_early = 1e7
alpha1 = 2.0
z_reion_sam = 9.0
M_c2_0 = 1e12
alpha2 = 1.0

zs_eps = [0.0, 1.0, 2.0, 4.0, 8.0, 12.0]
cmap_eps = plt.get_cmap('viridis_r')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for i, z_e in enumerate(zs_eps):
    c = cmap_eps(float(i) / (len(zs_eps) - 1))

    # eps_g1 (reionization suppression)
    if z_e > z_reion_sam:
        M_c1 = M_c1_early
    else:
        M_c1 = M_c1_0 * np.exp(-0.63 * z_e)
    eps1 = (1.0 + (2.0**(alpha1 / 3) - 1) * (Mh_eps / M_c1)**(-alpha1))**(-3.0 / alpha1)

    # eps_g2 (cooling cutoff)
    M_c2 = M_c2_0 * np.sqrt(cosmo.Ez(z_e)) * mass_so.deltaVir(z_e) / mass_so.deltaVir(0.0)
    eps2 = 1.0 - (1.0 + (2.0**(alpha2 / 3) - 1) * (Mh_eps / M_c2)**(-alpha2))**(-3.0 / alpha2)

    label = r'$z = %g$' % z_e
    axes[0].semilogx(Mh_eps, eps1, color=c, label=label)
    axes[1].semilogx(Mh_eps, eps2, color=c, label=label)
    axes[2].semilogx(Mh_eps, eps1 * eps2, color=c, lw=2, label=label)

for ax in axes:
    ax.set_xlabel(r'$M_{\rm halo}\ (M_\odot)$')
    ax.set_xlim(1e6, 1e15)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=7, labelspacing=0.2)

axes[0].set_ylabel('Efficiency')
axes[0].set_title(r'$\epsilon_1$ (Reionization)')
axes[1].set_title(r'$\epsilon_2$ (Cooling)')
axes[2].set_title(r'$\epsilon_1 \times \epsilon_2$ (Combined)')

plt.tight_layout()
plt.show()

In [ ]:
# Run the fiducial model for a grid of initial halo masses
z_ini = 30.0
Mh_ini_array = 10**np.linspace(6.4, 7.85, 30)

print(f'Running {len(Mh_ini_array)} models from z={z_ini} to z=0...')
t, z, a, M_hist, dM_hist = rt.evaluateModel(z_ini, Mh_ini_array)

# Present-day halo masses
Mh_z0 = M_hist[:, rt.IH, -1]
Ms_z0 = M_hist[:, rt.IS, -1]
Mg_z0 = M_hist[:, rt.IG, -1]
print(f'Halo mass range at z=0: {Mh_z0.min():.2e} - {Mh_z0.max():.2e} Msun')
print(f'Stellar mass range at z=0: {Ms_z0.min():.2e} - {Ms_z0.max():.2e} Msun')

## Exercise 1: Mass Evolution Histories

Plot the evolution of all four components ($M_h$, $M_g$, $M_*$, $M_Z$) as a function of redshift for four representative halos spanning the mass range. Identify:

1. When does star formation begin in each halo?
2. How does the gas fraction $M_g/M_h$ evolve?
3. At what redshift does feedback become important (gas mass starts declining)?

In [ ]:
# Exercise 1: Plot mass evolution for 4 representative halos
indices = [0, 10, 20, 29]  # indices into Mh_ini_array
labels_comp = [r'$M_h$', r'$M_g$', r'$M_*$', r'$M_Z$']
colors_comp = ['black', 'blue', 'red', 'green']

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for k, idx in enumerate(indices):
    ax = axes[k]
    ax.set_yscale('log')
    ax.set_xlabel('Redshift $z$')
    ax.set_ylabel(r'$M\ (M_\odot)$')
    ax.set_xlim(z[-1], 20)
    ax.invert_xaxis()

    # FILL IN: plot each component (halo, gas, stars, metals)
    for comp in range(4):
        ...

    ax.set_title(r'$M_h(z\!=\!0) = %.1e\ M_\odot$' % Mh_z0[idx])
    if k == 0:
        ax.legend(fontsize=8, loc='lower left')

plt.tight_layout()
plt.show()

## The Stellar-to-Halo Mass Relation (SHMR)

The SHMR is the central prediction of any galaxy formation model. It tells us how efficiently each halo converts its baryons into stars.

From our model, we extract $M_*(M_h)$ at $z = 0$ and compare to observational constraints from:

- Behroozi et al. (2013): abundance matching
- Kravtsov et al. (2018): updated abundance matching with scatter

## Exercise 2: SHMR and Star Formation Efficiency

1. Plot $M_*/M_h$ vs $M_h$ from the model and compare to Behroozi+2013 and Kravtsov+2018
2. Overplot the cosmic baryon fraction $f_b = \Omega_b / \Omega_m$
3. Identify the halo mass where star formation efficiency peaks

In [ ]:
# Exercise 2: SHMR from the semi-analytic model

# FILL IN: compute stellar-to-halo mass ratio from the model
shmr_model = ...

# Observational comparisons
Mvir_beh, shmr_beh = rt.shmrBehroozi13()
Mvir_kra, shmr_kra = rt.shmrKravtsov18()

fb = cosmo.Ob0 / cosmo.Om0

plt.figure(figsize=(5, 4.5))
plt.loglog()
plt.xlabel(r'$M_{\rm halo}\ (M_\odot)$')
plt.ylabel(r'$M_* / M_{\rm halo}$')
plt.xlim(1e8, 1e15)
plt.ylim(1e-6, 0.5)

# FILL IN: plot model SHMR
...

plt.plot(Mvir_beh, shmr_beh, '--', color='blue', lw=1.5, label='Behroozi+2013')
plt.plot(Mvir_kra, shmr_kra, '-.', color='orange', lw=1.5, label='Kravtsov+2018')
plt.axhline(fb, ls=':', color='gray', lw=0.8, label=r'$f_b = %.3f$' % fb)

plt.legend(fontsize=9)
plt.title('Stellar-to-Halo Mass Relation')
plt.tight_layout()
plt.show()

# Find peak efficiency
i_peak = np.argmax(shmr_model)
print(f'Peak efficiency: M*/Mh = {shmr_model[i_peak]:.4f} at Mh = {Mh_z0[i_peak]:.2e} Msun')
print(f'Peak efficiency / f_b = {shmr_model[i_peak] / fb:.2f}')

## Demonstration: Gas Fraction and Mass-Metallicity Relation

Beyond the SHMR, the SAM makes testable predictions for gas fractions and metallicities:

- Gas fraction $f_{\rm gas} = M_g / (M_g + M_*)$: low-mass galaxies retain more of their gas (star formation is inefficient), while massive galaxies have converted most of their gas into stars.
- Mass-metallicity relation (MZR): more massive galaxies are more metal-rich because they retain more of their enriched gas (less is lost to winds). This is one of the tightest observed correlations in galaxy evolution (Tremonti et al. 2004).

Both emerge naturally from the coupled gas/star/metal ODEs in the SAM (adapted from CFN Chapter 8).

In [ ]:
# Gas fraction and mass-metallicity relation from the SAM at z=0
Mz_z0 = M_hist[:, rt.IZ, -1]

# Gas fraction: f_gas = Mg / (Mg + M*)
fgas = Mg_z0 / (Mg_z0 + Ms_z0)

# Metallicity in solar units: Z = (MZ/Mg) / Z_solar
Z_solar = 0.0134
metallicity = (Mz_z0 / Mg_z0) / Z_solar

# Mask out halos with negligible stellar mass
mask_met = (Ms_z0 > 1e5) & (Mg_z0 > 0)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Left: gas fraction vs stellar mass
ax1 = axes[0]
ax1.semilogx(Ms_z0[mask_met], fgas[mask_met], 'ko-', ms=4, lw=1.5)
ax1.set_xlabel(r'$M_*\ (M_\odot)$')
ax1.set_ylabel(r'$f_{\rm gas} = M_g / (M_g + M_*)$')
ax1.set_xlim(1e5, 1e12)
ax1.set_ylim(0, 1.05)
ax1.axhline(0.5, ls=':', color='gray', lw=0.8)
ax1.set_title('Gas Fraction at $z=0$')

# Right: mass-metallicity relation
ax2 = axes[1]
ax2.loglog(Ms_z0[mask_met], metallicity[mask_met], 'ko-', ms=4, lw=1.5,
           label='SAM (fiducial)')
ax2.set_xlabel(r'$M_*\ (M_\odot)$')
ax2.set_ylabel(r'$Z / Z_\odot$')
ax2.set_xlim(1e5, 1e12)
ax2.set_ylim(1e-3, 10)
ax2.axhline(1.0, ls=':', color='gray', lw=0.8, label=r'$Z_\odot$')
ax2.legend(fontsize=9)
ax2.set_title('Mass–Metallicity Relation at $z=0$')

plt.tight_layout()
plt.show()

## The Role of Feedback

The SHMR shape is determined by the interplay of physics:

- Low-mass suppression: reionization ($\epsilon_1$) prevents gas accretion onto small halos, and strong stellar winds ($\eta \gg 1$) eject gas efficiently
- High-mass suppression: cooling becomes inefficient ($\epsilon_2 \to 0$) above $M_h \sim 10^{12}\,M_\odot$ (AGN feedback in reality)
- Peak efficiency: at $M_h \sim 10^{12}\,M_\odot$ (MW mass), both suppression mechanisms are weak

To understand the contribution of each mechanism, we can turn them off one by one.

## Exercise 3: Turning Off Physics

Run three additional models with one physics component disabled each:

1. No reionization: `model_reion='none'`
2. No cooling cutoff: `model_cooling='none'`
3. No feedback: `model_wind='none'`

Compare the resulting SHMRs to understand which mechanism shapes which part of the relation.

In [ ]:
# Exercise 3: Turn off physics components one by one

# FILL IN: run model with no reionization
_, _, _, M_no_reion, _ = ...

# FILL IN: run model with no cooling cutoff
_, _, _, M_no_cool, _ = ...

# FILL IN: run model with no feedback/winds
_, _, _, M_no_wind, _ = ...

# Extract z=0 SHMRs
Mh_z0_nr = M_no_reion[:, rt.IH, -1]
Ms_z0_nr = M_no_reion[:, rt.IS, -1]

Mh_z0_nc = M_no_cool[:, rt.IH, -1]
Ms_z0_nc = M_no_cool[:, rt.IS, -1]

Mh_z0_nw = M_no_wind[:, rt.IH, -1]
Ms_z0_nw = M_no_wind[:, rt.IS, -1]

# Plot comparison
plt.figure(figsize=(6, 5))
plt.loglog()
plt.xlabel(r'$M_{\rm halo}\ (M_\odot)$')
plt.ylabel(r'$M_* / M_{\rm halo}$')
plt.xlim(1e8, 1e15)
plt.ylim(1e-6, 0.5)

plt.plot(Mh_z0, Ms_z0 / Mh_z0, 'k-', lw=2, label='Fiducial')

# FILL IN: plot the 3 variant SHMRs
...

plt.plot(Mvir_beh, shmr_beh, ':', color='gray', lw=1, label='Behroozi+2013')
plt.axhline(fb, ls=':', color='gray', lw=0.5)

plt.legend(fontsize=9)
plt.title('Impact of Physics on the SHMR')
plt.tight_layout()
plt.show()

## Demonstration: SHMR Evolution with Redshift

The SHMR evolves with cosmic time. At higher redshifts, halos are less massive and the efficiency curve shifts. The SAM naturally predicts this evolution since it tracks each halo from $z = 30$ to $z = 0$.

Below we extract the SHMR from the model at several redshifts and compare to the Behroozi et al. (2013) fitting function, which was calibrated across $0 < z < 8$. The overall shape is preserved but the peak efficiency and characteristic mass evolve (adapted from CFN Chapter 8).

In [ ]:
# SHMR evolution with redshift: SAM vs Behroozi+2013
zs_shmr = [0.0, 0.5, 1.0, 2.0, 4.0]
cmap_shmr = plt.get_cmap('viridis')

plt.figure(figsize=(5.5, 4.5))
plt.loglog()
plt.xlabel(r'$M_{\rm halo}\ (M_\odot)$')
plt.ylabel(r'$M_* / M_{\rm halo}$')
plt.xlim(1e9, 1e15)
plt.ylim(1e-5, 0.5)

for i, z_s in enumerate(zs_shmr):
    c = cmap_shmr(float(i) / (len(zs_shmr) - 1))
    # Find closest time index in the model output
    i_z = np.argmin(np.abs(z - z_s))
    Mh_at_z = M_hist[:, rt.IH, i_z]
    Ms_at_z = M_hist[:, rt.IS, i_z]
    mask_sam = (Ms_at_z > 0) & (Mh_at_z > 1e9)

    # SAM prediction
    plt.plot(Mh_at_z[mask_sam], Ms_at_z[mask_sam] / Mh_at_z[mask_sam],
             '-', color=c, lw=1.5, label=r'SAM $z=%g$' % z_s)

    # Behroozi+2013 at same redshift
    Mvir_beh_z = 10**np.linspace(10, 15, 100)
    mstar_beh_z = rt.mstarBehroozi13(Mvir_beh_z, z_s)
    mask_beh = mstar_beh_z > 1e7
    plt.plot(Mvir_beh_z[mask_beh], mstar_beh_z[mask_beh] / Mvir_beh_z[mask_beh],
             '--', color=c, lw=0.8, alpha=0.7)

plt.axhline(fb, ls=':', color='gray', lw=0.5)
plt.plot([], [], '--', color='gray', lw=0.8, label='Behroozi+2013')
plt.legend(fontsize=8, ncol=2)
plt.title('SHMR: Redshift Evolution')
plt.tight_layout()
plt.show()

## Interpretation

The comparison reveals:

- Without reionization: low-mass halos form too many stars — the low-mass end of the SHMR rises dramatically. Reionization is essential to explain the observed suppression below $M_h \sim 10^{10}\,M_\odot$.
- Without cooling cutoff: high-mass halos convert too many baryons into stars. In reality, AGN feedback and long cooling times prevent this.
- Without feedback (winds): even intermediate-mass halos form too many stars. Stellar winds remove gas and regulate the overall efficiency.

All three mechanisms are needed to reproduce the observed SHMR. This is one of the central results of galaxy formation theory.

## Summary

1. Semi-analytic models track the co-evolution of dark matter, gas, stars, and metals via coupled ODEs with physically motivated sub-models.
2. The SHMR has a characteristic peaked shape: maximum efficiency at $M_h \sim 10^{12}\,M_\odot$ ($\sim 20$–$25\%$ of available baryons converted to stars).
3. Three feedback mechanisms shape the SHMR:
   - Reionization suppresses low-mass galaxies
   - Cooling inefficiency suppresses high-mass galaxies
   - Stellar winds regulate intermediate-mass galaxies
4. No single mechanism is sufficient — the observed galaxy population requires all three operating simultaneously. This “conspiracy” of feedback mechanisms is a fundamental challenge for galaxy formation theory.